# Explore here

In [ ]:
import pandas as pd

# Rutas de los archivos
file_df1   = "../data/processed/dataset_completo.csv"
file_hot   = "../data/processed/hotels_with_distance_dataset.csv"
file_vuel  = "../data/processed/merged_flight_dataset.csv"

# Parámetros
chunksize    = 200_000
fecha_col    = "fecha"
fecha_umbral = pd.Timestamp("2022-01-01")

# Cargo hoteles y vuelos completos (deberían caber en RAM)
df_hoteles = pd.read_csv(file_hot)
df_vuelos  = pd.read_csv(file_vuel)

# 1) Procesar dataset_completo por chunks
merged_acc = []
for chunk in pd.read_csv(file_df1, parse_dates=[fecha_col], chunksize=chunksize):
    # 1.a) Filtrar por fecha reciente
    recent = chunk[chunk[fecha_col] >= fecha_umbral]
    if recent.empty:
        continue

    # 1.b) Merge con hoteles
    h = pd.merge(
        recent,
        df_hoteles,
        how="left",
        left_on="ciudad",
        right_on="destination_city"
    )
    # 1.c) Merge con vuelos
    hv = pd.merge(
        h,
        df_vuelos,
        how="left",
        left_on="ciudad",
        right_on="destination_city"
    )
    merged_acc.append(hv)

# 2) Concatenar los trozos ya filtrados y mergeados
total_data = pd.concat(merged_acc, ignore_index=True)
del merged_acc

# 3) Ordenar por fecha descendente
total_data = total_data.sort_values(fecha_col, ascending=False)

# Ahora total_data está listo para el muestreo estratificado
print(f"Total registros tras merge y filtro: {len(total_data)}")

In [ ]:
# Parámetros de muestreo
N = 240_000

# 4) Calcular cuántos registros le tocan a cada ciudad
conteo_por_ciudad  = total_data["ciudad"].value_counts()
proporciones      = conteo_por_ciudad / conteo_por_ciudad.sum()
muestra_por_ciudad = (proporciones * N).round().astype(int)

# 5) Función para tomar los n más recientes de cada grupo
def take_top_n(group):
    n = muestra_por_ciudad[group.name]
    return group.head(n)

# 6) Aplicar muestreo estratificado
sample_data = (
    total_data
    .groupby("ciudad", group_keys=False)
    .apply(take_top_n)
)

# Ajustar si no suma exactamente N (por redondeos)
if len(sample_data) > N:
    sample_data = sample_data.head(N)
elif len(sample_data) < N:
    deficit = N - len(sample_data)
    # Tomar los siguientes más recientes, sin repetir índices
    extras = total_data.drop(sample_data.index).head(deficit)
    sample_data = pd.concat([sample_data, extras], ignore_index=True)

print(f"✅ Muestreo completo: {len(sample_data)} filas obtenidas.")

In [ ]:
sample_data.to_csv(file_df1.replace("dataset_completo.csv", "total_data_240k.csv"), index=False)
sample_data.to_csv(file_df1.replace("dataset_completo.csv", "total_data_240k.csv.gz"),
                   index=False, compression="gzip")
print("✅ Archivos guardados.")